# AIML339 Galaxy Morphology Classification Comparison Project

donaldseth - 300665568

In [49]:
# Imports

import pandas as pd
import numpy as np

from zipfile import ZipFile
from PIL import Image
from io import BytesIO

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

In [27]:
# Paths

LABELS_PATH = "training_solutions_rev1.csv"
IMAGES_PATH = "images_training_rev1.zip"

In [28]:
# Load Labels

labels = pd.read_csv(LABELS_PATH)

print("Total galaxies:", len(labels))
print("Number of classes:", len(labels.columns) - 1)

Total galaxies: 61578
Number of classes: 37


In [29]:
# Labels Subset

N_GALAXIES = 5000

labels_subset = labels.sample(
    n=N_GALAXIES,
    random_state=42
).copy()

labels_subset["filename"] = (
    labels_subset["GalaxyID"].astype(str)
)

labels_subset.head()

,GalaxyID,Class1.1,Class1.2,Class1.3,Class2.1,Class2.2,Class3.1,Class3.2,Class4.1,Class4.2,...,Class10.1,Class10.2,Class10.3,Class11.1,Class11.2,Class11.3,Class11.4,Class11.5,Class11.6,filename
30888,553402,0.018765,0.955090,0.026146,0.078768,0.876322,0.118995,0.757327,0.590601,0.285720,...,0.000000,0.136383,0.454219,0.193254,0.289966,0.000000,0.0,0.000000,0.107381,553402
23964,449744,0.806705,0.140380,0.052915,0.000000,0.140380,0.000000,0.140380,0.000000,0.140380,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,449744
53475,879640,0.662354,0.337646,0.000000,0.197076,0.140570,0.000000,0.140570,0.044101,0.096468,...,0.044101,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.044101,879640
16284,340485,0.092952,0.907048,0.000000,0.058283,0.848765,0.042052,0.806713,0.642360,0.206404,...,0.496498,0.134098,0.011765,0.000000,0.373704,0.010232,0.0,0.013647,0.244777,340485
60574,985545,0.144641,0.848689,0.006670,0.028017,0.820672,0.484659,0.336014,0.267777,0.552895,...,0.000000,0.219558,0.048219,0.000000,0.195449,0.000000,0.0,0.000000,0.072328,985545


In [30]:
# Collect from .zip file

with ZipFile(IMAGES_PATH, "r") as zip_file:
    
    images = []

    for galaxy_id in labels_subset["filename"]:
        filename = f"images_training_rev1/{galaxy_id}.jpg"
        image_data = zip_file.read(filename)

        image = Image.open(BytesIO(image_data))
        images.append(image.copy())

print("Images collected:", len(images))

Images collected: 5000


In [31]:
print(labels_subset.columns.tolist())

['GalaxyID', 'Class1.1', 'Class1.2', 'Class1.3', 'Class2.1', 'Class2.2', 'Class3.1', 'Class3.2', 'Class4.1', 'Class4.2', 'Class5.1', 'Class5.2', 'Class5.3', 'Class5.4', 'Class6.1', 'Class6.2', 'Class7.1', 'Class7.2', 'Class7.3', 'Class8.1', 'Class8.2', 'Class8.3', 'Class8.4', 'Class8.5', 'Class8.6', 'Class8.7', 'Class9.1', 'Class9.2', 'Class9.3', 'Class10.1', 'Class10.2', 'Class10.3', 'Class11.1', 'Class11.2', 'Class11.3', 'Class11.4', 'Class11.5', 'Class11.6', 'filename']


In [32]:
# Show the response probabilities for the first two Galaxy Zoo tasks
columns = [
    "Class1.1", "Class1.2", "Class1.3",
    "Class2.1", "Class2.2"
]

print(labels_subset[columns].head(10))

       Class1.1  Class1.2  Class1.3  Class2.1  Class2.2
30888  0.018765  0.955090  0.026146  0.078768  0.876322
23964  0.806705  0.140380  0.052915  0.000000  0.140380
53475  0.662354  0.337646  0.000000  0.197076  0.140570
16284  0.092952  0.907048  0.000000  0.058283  0.848765
60574  0.144641  0.848689  0.006670  0.028017  0.820672
17962  0.553336  0.374126  0.072539  0.000000  0.374126
19482  0.529935  0.379124  0.090940  0.000000  0.379124
24973  0.518825  0.456786  0.024389  0.029735  0.427051
59184  0.130019  0.869981  0.000000  0.000000  0.869981
32165  0.438559  0.512630  0.048810  0.068876  0.443754


In [33]:
# Keep galaxies whose most likely Task 01 response is not star/artifact
task1_columns = ["Class1.1", "Class1.2", "Class1.3"]

task1_labels = labels_subset[task1_columns].idxmax(axis=1)

classification_data = labels_subset[
    task1_labels != "Class1.3"
].copy()

# The three final classes follow our simplified decision tree
class_columns = ["Class1.1", "Class2.1", "Class2.2"]

class_names = [
    "Smooth",
    "Featured edge-on",
    "Featured not edge-on"
]

# Assign each galaxy the class with the largest weighted vote
classification_data["target"] = (
    classification_data[class_columns]
    .to_numpy()
    .argmax(axis=1)
)

# Check how many galaxies belong to each class
print("Total galaxies:", len(classification_data))

print(
    classification_data["target"]
    .value_counts()
    .sort_index()
    .rename(index=dict(enumerate(class_names)))
)

Total galaxies: 4993
target
Smooth                  2256
Featured edge-on         511
Featured not edge-on    2226
Name: count, dtype: int64


Dataset splitting: The initial prototype used a random 80/20 split followed by a training/validation split. For the final three-class experiments, stratified splitting is used to preserve the class distribution across training, validation and test sets.

In [34]:
# Split the labelled galaxies into training/validation and test sets
train_val, test = train_test_split(
    classification_data,
    test_size=0.20,
    random_state=42,
    stratify=classification_data["target"]
)

# Split the remaining data into training and validation sets
train, validation = train_test_split(
    train_val,
    test_size=0.20,
    random_state=42,
    stratify=train_val["target"]
)

# Check the number of galaxies in each dataset
print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

# Check the class distribution in each dataset
for name, data in [
    ("Training", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{name} class counts:")
    print(data["target"].value_counts().sort_index())

Training: 3195
Validation: 799
Test: 999

Training class counts:
target
0    1443
1     327
2    1425
Name: count, dtype: int64

Validation class counts:
target
0    361
1     82
2    356
Name: count, dtype: int64

Test class counts:
target
0    452
1    102
2    445
Name: count, dtype: int64


In [36]:
# Image Transformation

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# For greyscale:

#transform = transforms.Compose([
#    transforms.Grayscale(num_output_channels=1),
#    transforms.Resize((64, 64)),
#    transforms.ToTensor()
#])

In [37]:
class GalaxyDataset(Dataset):
    def __init__(self, data, transform=None):
        # Store the dataset and image transformations
        self.data = data.reset_index(drop=True)
        self.transform = transform

        # Open the Galaxy Zoo image archive
        self.zip_file = ZipFile(IMAGES_PATH, "r")

    def __len__(self):
        # Return the number of galaxies in the dataset
        return len(self.data)

    def __getitem__(self, index):
        # Get the galaxy information for this row
        row = self.data.iloc[index]

        # Locate and read the image inside the ZIP archive
        filename = f"images_training_rev1/{row['filename']}.jpg"
        image_data = self.zip_file.read(filename)

        # Convert the image to RGB
        image = Image.open(BytesIO(image_data)).convert("RGB")

        # Resize and convert the image into a tensor
        if self.transform:
            image = self.transform(image)

        # Return the new three-class label as an integer
        label = torch.tensor(int(row["target"]), dtype=torch.long)

        return image, label

In [38]:
# Dataset Creation

train_dataset = GalaxyDataset(train, transform)
validation_dataset = GalaxyDataset(validation, transform)
test_dataset = GalaxyDataset(test, transform)

print("Datasets created")

Datasets created


In [39]:
# DataLoaders

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [40]:
images, labels_batch = next(iter(train_loader))

print("Image shape:", images.shape)
print("Label shape:", labels_batch.shape)

Image shape: torch.Size([32, 3, 64, 64])
Label shape: torch.Size([32])


In [41]:
# End-to-end CNN for classifying galaxies into three categories

class GalaxyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Extract visual features from the galaxy images
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Use the extracted features to predict one of three classes
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [42]:
# GPU

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

Using: cuda
NVIDIA GeForce RTX 5070 Laptop GPU


In [43]:
# Create the CNN and move it to the GPU if available

model = GalaxyCNN().to(device)

print(model)

GalaxyCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [44]:
# Loss and optimizer

# Cross-entropy is suitable for classification into three classes

criterion = nn.CrossEntropyLoss()

# Adam updates the CNN's weights during training

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [45]:
# Get one batch of galaxy images
images, labels = next(iter(train_loader))

# Move the images to the same device as the CNN
images = images.to(device)

# Run the images through the CNN
with torch.no_grad():
    outputs = model(images)

print("Output shape:", outputs.shape)
print("Label shape:", labels.shape)

Output shape: torch.Size([32, 3])
Label shape: torch.Size([32])


## End-to-End CNN Training and Validation

In [46]:
# Train the three-class end-to-end CNN

num_epochs = 10

train_losses = []
validation_losses = []

for epoch in range(num_epochs):
    # Training mode enables weight updates
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Predict class scores and calculate the error
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Calculate gradients and update model weights
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    # Evaluate on validation data without updating weights
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

    epoch_val_loss = running_val_loss / len(validation_dataset)
    validation_losses.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Training loss: {epoch_train_loss:.4f} | "
        f"Validation loss: {epoch_val_loss:.4f}"
    )

Epoch 1/10 | Training loss: 0.9446 | Validation loss: 0.8851
Epoch 2/10 | Training loss: 0.8739 | Validation loss: 0.7854
Epoch 3/10 | Training loss: 0.7330 | Validation loss: 0.6587
Epoch 4/10 | Training loss: 0.6538 | Validation loss: 0.6865
Epoch 5/10 | Training loss: 0.6147 | Validation loss: 0.6382
Epoch 6/10 | Training loss: 0.6017 | Validation loss: 0.6209
Epoch 7/10 | Training loss: 0.5721 | Validation loss: 0.6188
Epoch 8/10 | Training loss: 0.5468 | Validation loss: 0.6293
Epoch 9/10 | Training loss: 0.5278 | Validation loss: 0.6096
Epoch 10/10 | Training loss: 0.5098 | Validation loss: 0.6107


In [50]:
# Evaluate the end-to-end CNN on validation data

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in validation_loader:
        images = images.to(device)

        # Predict the most likely class for each image
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

class_names = [
    "Smooth",
    "Featured edge-on",
    "Featured not edge-on"
]

# Overall accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Validation accuracy: {accuracy:.4f}")

# Precision, recall and F1-score for each class
print("\nClassification report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2],
        target_names=class_names,
        zero_division=0
    )
)

# Show which classes are being confused
print("\nConfusion matrix:")
print(confusion_matrix(all_labels, all_predictions, labels=[0, 1, 2]))

Validation accuracy: 0.7409

Classification report:
                      precision    recall  f1-score   support

              Smooth       0.82      0.71      0.76       361
    Featured edge-on       0.67      0.59      0.62        82
Featured not edge-on       0.70      0.81      0.75       356

            accuracy                           0.74       799
           macro avg       0.73      0.70      0.71       799
        weighted avg       0.75      0.74      0.74       799


Confusion matrix:
[[255   7  99]
 [  7  48  27]
 [ 50  17 289]]


In [20]:
# Test

model.eval()

test_loss = 0.0

with torch.no_grad():
    
    for images, labels_batch in test_loader:
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        predictions = model(images)
        loss = criterion(predictions, labels_batch)
        
        test_loss += loss.item()

test_loss /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.0184


In [21]:
# Look at one galaxy

image, true_labels = test_dataset[0]

model.eval()

with torch.no_grad():
    prediction = model(
        image.unsqueeze(0).to(device)
    )

prediction = prediction.cpu().numpy()[0]
true_labels = true_labels.numpy()

print("True probabilities:")
print(true_labels)

print("\nPredicted probabilities:")
print(prediction)

True probabilities:
[0.127217   0.847839   0.024944   0.36012724 0.48771176 0.05610148
 0.43161026 0.21759942 0.27011234 0.04478072 0.35924703 0.
 0.08368354 0.295699   0.704301   0.         0.02694125 0.10027575
 0.         0.08087368 0.07423228 0.11761516 0.02297818 0.
 0.         0.118076   0.03941593 0.20263532 0.1229798  0.09461963
 0.         0.1424334  0.         0.         0.         0.
 0.07516602]

Predicted probabilities:
[ 0.1644566   0.79207885  0.00930578  0.4238228   0.38144544  0.08669539
  0.2934673   0.20878229  0.15178488  0.03049997  0.2109208   0.12435368
 -0.00202131  0.09813224  0.86294144 -0.07098898  0.04582849  0.17632674
  0.05262552  0.01032387  0.0236659   0.02906393 -0.0021978   0.01349339
  0.00870462  0.22971736  0.04767209  0.1517778   0.13215111  0.08491918
  0.04269323  0.00809979  0.08427066  0.02739083  0.01186273 -0.00559747
  0.07743451]
